# 🐼 Pandas — Complete Detailed Guide (Beginner → Interview Ready)

**How to use:** run cells top to bottom. Every section = **Concept → Code → 🎯 Interview Points**.
All data is created inside the notebook, so no external files are needed.

## Table of Contents
1. What is Pandas? Series
2. DataFrame creation
3. Inspecting data
4. Selecting data (`[]`, `loc`, `iloc`, boolean, `query`)
5. Adding, modifying, renaming, dropping
6. Data types & conversion
7. Missing data
8. Duplicates
9. String operations (`.str`)
10. Date & time (`.dt`, resample)
11. Sorting & ranking
12. `apply`, `map`, vectorization
13. GroupBy (split-apply-combine)
14. Pivot, crosstab, melt, stack/unstack
15. Combining data: `merge`, `join`, `concat`
16. Window functions, `shift`, `diff`, `pct_change`
17. MultiIndex
18. Binning & counting (`cut`, `qcut`, `value_counts`)
19. Reading & writing files
20. Performance & memory optimization
21. Pitfalls (SettingWithCopy, Copy-on-Write, chaining)
22. Mini project: sales analysis
23. Interview Q&A (rapid-fire)
24. Practice problems with solutions
25. Cheat sheet

## 1. What is Pandas? — Series

**Pandas** is the main Python library for **tabular, labelled data** (like Excel/SQL tables in code). Built on top of NumPy.

| Structure | Description |
|---|---|
| **Series** | 1-D labelled array (one column) |
| **DataFrame** | 2-D labelled table (collection of Series sharing one index) |
| **Index** | Immutable labels for rows (or columns) |

### 🎯 Interview Points
- Pandas = NumPy + **labels** + **missing-data handling** + **mixed dtypes per column** + group/merge/time-series tools.
- Each DataFrame column is a Series with its **own dtype**; a NumPy array has a single dtype.

In [3]:
import numpy as np
import pandas as pd
print("pandas", pd.__version__, "| numpy", np.__version__)

# Series from list, with custom index
s = pd.Series([10, 20, 30, 40], index=["a", "b", "c", "d"], name="scores")
print(s)
print("\nvalues:", s.values, "| index:", list(s.index), "| dtype:", s.dtype)
print("s['b'] =", s["b"], "| s.iloc[0] =", s.iloc[0])

# Series from dict
pop = pd.Series({"Mumbai": 20.4, "Delhi": 31.2, "Pune": 7.4})
print(pop)

# Vectorized ops & index ALIGNMENT
s1 = pd.Series([1, 2, 3], index=["a", "b", "c"])
s2 = pd.Series([10, 20, 30], index=["b", "c", "d"])
print("\nAligned addition (NaN where label missing):\n", s1 + s2)
print("\nWith fill_value:\n", s1.add(s2, fill_value=0))

pandas 3.0.3 | numpy 2.4.6
a    10
b    20
c    30
d    40
Name: scores, dtype: int64

values: [10 20 30 40] | index: ['a', 'b', 'c', 'd'] | dtype: int64
s['b'] = 20 | s.iloc[0] = 10
Mumbai    20.4
Delhi     31.2
Pune       7.4
dtype: float64

Aligned addition (NaN where label missing):
 a     NaN
b    12.0
c    23.0
d     NaN
dtype: float64

With fill_value:
 a     1.0
b    12.0
c    23.0
d    30.0
dtype: float64


### 🎯 Interview Points
- **Index alignment**: arithmetic between Series/DataFrames aligns on **labels**, not positions. Non-matching labels give `NaN`. Use `.add(..., fill_value=0)` to avoid it. This is a favourite question.
- A Series has `.index` and `.values` (`.to_numpy()` is the recommended way to get the array).

## 2. Creating a DataFrame

In [4]:
# From dict of lists (most common)
df = pd.DataFrame({
    "name":   ["Asha", "Ben", "Chitra", "Dev", "Esha", "Farid"],
    "age":    [25, 32, 28, 45, 38, 29],
    "city":   ["Mumbai", "Delhi", "Pune", "Mumbai", "Pune", "Delhi"],
    "salary": [50000, 72000, 61000, 120000, 98000, 58000],
    "dept":   ["IT", "HR", "IT", "Finance", "IT", "HR"],
})
df

,name,age,city,salary,dept
0,Asha,25,Mumbai,50000,IT
1,Ben,32,Delhi,72000,HR
2,Chitra,28,Pune,61000,IT
3,Dev,45,Mumbai,120000,Finance
4,Esha,38,Pune,98000,IT
5,Farid,29,Delhi,58000,HR


In [5]:
# From list of dicts (typical JSON/API style)
pd.DataFrame([{"a": 1, "b": 2}, {"a": 3, "c": 4}])   # missing keys -> NaN

,a,b,c
0,1,2.0,NaN
1,3,NaN,4.0


In [6]:
# From NumPy array with labels
pd.DataFrame(np.arange(6).reshape(2, 3), columns=list("ABC"), index=["r1", "r2"])

,A,B,C
r1,0,1,2
r2,3,4,5


In [7]:
# Other constructors
print(pd.DataFrame.from_records([(1, "x"), (2, "y")], columns=["id", "val"]))
print(pd.date_range("2024-01-01", periods=4, freq="D"))
print(pd.DataFrame({"a": range(3)}, index=pd.Index(list("xyz"), name="key")))

   id val
0   1   x
1   2   y
DatetimeIndex(['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04'], dtype='datetime64[us]', freq='D')
     a
key   
x    0
y    1
z    2


## 3. Inspecting Data

In [8]:
print(df.shape, df.ndim, df.size)
print(df.columns.tolist(), "|", df.index)
print(df.head(3)); print(df.tail(2)); print(df.sample(2, random_state=1))

(6, 5) 2 30
['name', 'age', 'city', 'salary', 'dept'] | RangeIndex(start=0, stop=6, step=1)
     name  age    city  salary dept
0    Asha   25  Mumbai   50000   IT
1     Ben   32   Delhi   72000   HR
2  Chitra   28    Pune   61000   IT
    name  age   city  salary dept
4   Esha   38   Pune   98000   IT
5  Farid   29  Delhi   58000   HR
     name  age   city  salary dept
2  Chitra   28   Pune   61000   IT
1     Ben   32  Delhi   72000   HR


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   name    6 non-null      str  
 1   age     6 non-null      int64
 2   city    6 non-null      str  
 3   salary  6 non-null      int64
 4   dept    6 non-null      str  
dtypes: int64(2), str(3)
memory usage: 444.0 bytes


In [10]:
print(df.describe())                 # numeric summary
print(df.describe(include="all"))    # include object columns
print(df.dtypes)
print(df["city"].nunique(), df["city"].unique())
print(df.memory_usage(deep=True))

             age         salary
count   6.000000       6.000000
mean   32.833333   76500.000000
std     7.413951   27053.650401
min    25.000000   50000.000000
25%    28.250000   58750.000000
50%    30.500000   66500.000000
75%    36.500000   91500.000000
max    45.000000  120000.000000
        name        age    city         salary dept
count      6   6.000000       6       6.000000    6
unique     6        NaN       3            NaN    3
top     Asha        NaN  Mumbai            NaN   IT
freq       1        NaN       2            NaN    3
mean     NaN  32.833333     NaN   76500.000000  NaN
std      NaN   7.413951     NaN   27053.650401  NaN
min      NaN  25.000000     NaN   50000.000000  NaN
25%      NaN  28.250000     NaN   58750.000000  NaN
50%      NaN  30.500000     NaN   66500.000000  NaN
75%      NaN  36.500000     NaN   91500.000000  NaN
max      NaN  45.000000     NaN  120000.000000  NaN
name        str
age       int64
city        str
salary    int64
dept        str
dtype: o

### 🎯 Interview Points
- `info()` prints, returns `None`. Shows non-null counts & dtypes & memory.
- `describe()` only numeric by default; `include="all"` or `include="object"` for strings.
- `shape` is an attribute (no parentheses); `head()` is a method.
- `nunique()` ignores NaN by default (`dropna=True`); `len(df)` = rows.

## 4. Selecting Data ⭐ (loc vs iloc is asked in almost every interview)

In [11]:
# Columns
print(df["name"].head(2))                 # Series
print(df[["name", "salary"]].head(2))     # DataFrame (note double brackets)

0    Asha
1     Ben
Name: name, dtype: str
   name  salary
0  Asha   50000
1   Ben   72000


In [12]:
# .loc  -> LABEL based, slice end is INCLUSIVE
# .iloc -> POSITION based, slice end is EXCLUSIVE
print(df.loc[0:2, ["name", "age"]])       # rows 0,1,2 (inclusive!)
print(df.iloc[0:2, 0:2])                  # rows 0,1 / cols 0,1
print(df.iloc[-1])                        # last row
print(df.at[1, "name"], df.iat[1, 0])     # fast scalar access

     name  age
0    Asha   25
1     Ben   32
2  Chitra   28
   name  age
0  Asha   25
1   Ben   32
name      Farid
age          29
city      Delhi
salary    58000
dept         HR
Name: 5, dtype: object
Ben Ben


In [ ]:
# Boolean filtering
print(df[df["salary"] > 60000])
print(df[(df["age"] > 28) & (df["city"] == "Pune")])        # & | ~  with parentheses
print(df[df["city"].isin(["Pune", "Delhi"])])
print(df[df["age"].between(28, 38)])
print(df[~df["dept"].eq("IT")])
print(df.query("salary > 60000 and dept == 'IT'"))
min_sal = 55000
print(df.query("salary > @min_sal")["name"].tolist())     # @ refers to Python variables

     name  age    city  salary     dept
1     Ben   32   Delhi   72000       HR
2  Chitra   28    Pune   61000       IT
3     Dev   45  Mumbai  120000  Finance
4    Esha   38    Pune   98000       IT
   name  age  city  salary dept
4  Esha   38  Pune   98000   IT
     name  age   city  salary dept
1     Ben   32  Delhi   72000   HR
2  Chitra   28   Pune   61000   IT
4    Esha   38   Pune   98000   IT
5   Farid   29  Delhi   58000   HR
     name  age   city  salary dept
1     Ben   32  Delhi   72000   HR
2  Chitra   28   Pune   61000   IT
4    Esha   38   Pune   98000   IT
5   Farid   29  Delhi   58000   HR
    name  age    city  salary     dept
1    Ben   32   Delhi   72000       HR
3    Dev   45  Mumbai  120000  Finance
5  Farid   29   Delhi   58000       HR
     name  age  city  salary dept
2  Chitra   28  Pune   61000   IT
4    Esha   38  Pune   98000   IT
['Ben', 'Chitra', 'Dev', 'Esha', 'Farid']


In [14]:
# Setting a custom index
d2 = df.set_index("name")
print(d2.loc["Asha"])
print(d2.loc["Ben":"Dev", "salary"])        # label slicing is inclusive
print(d2.reset_index().head(2))

age           25
city      Mumbai
salary     50000
dept          IT
Name: Asha, dtype: object
name
Ben        72000
Chitra     61000
Dev       120000
Name: salary, dtype: int64
   name  age    city  salary dept
0  Asha   25  Mumbai   50000   IT
1   Ben   32   Delhi   72000   HR


In [15]:
# Selecting columns by dtype / filter / pattern
print(df.select_dtypes(include="number").columns.tolist())
print(df.filter(like="a").columns.tolist(), df.filter(regex="^s").columns.tolist())
# Conditional values
df["level"] = np.where(df["salary"] > 70000, "High", "Normal")
print(df[["name", "level"]])

['age', 'salary']
['name', 'age', 'salary'] ['salary']
     name   level
0    Asha  Normal
1     Ben    High
2  Chitra  Normal
3     Dev    High
4    Esha    High
5   Farid  Normal


### 🎯 Interview Points
| | `loc` | `iloc` |
|---|---|---|
| Based on | labels | integer positions |
| Slice end | **inclusive** | **exclusive** |
| Boolean mask | ✅ | ✅ (needs array, not Series with index) |

- `df["a"]` → Series; `df[["a"]]` → DataFrame.
- `df[1:3]` slices **rows by position**, `df["a"]` selects **column** — `[]` is overloaded.
- Use `&`, `|`, `~` with **parentheses** (not `and/or/not`).
- `at`/`iat` are faster than `loc`/`iloc` for single values.
- After filtering the index is **not** reset; use `.reset_index(drop=True)`.
- Prefer a single `.loc[mask, col] = value` to assign (avoids chained assignment).

## 5. Adding, Modifying, Renaming & Dropping

In [16]:
d = df.copy()
d["bonus"] = d["salary"] * 0.1                       # new column (vectorized)
d["tax"] = np.where(d["salary"] > 70000, 0.3, 0.2) * d["salary"]
d = d.assign(net=lambda x: x["salary"] - x["tax"])   # method chaining style
d.loc[d["name"] == "Ben", "age"] = 33                # safe conditional update
d.insert(1, "id", range(100, 106))                   # insert at position
print(d.head(3))

     name   id  age    city  salary dept   level   bonus      tax      net
0    Asha  100   25  Mumbai   50000   IT  Normal  5000.0  10000.0  40000.0
1     Ben  101   33   Delhi   72000   HR    High  7200.0  21600.0  50400.0
2  Chitra  102   28    Pune   61000   IT  Normal  6100.0  12200.0  48800.0


In [17]:
d = d.rename(columns={"salary": "ctc", "name": "employee"})
d.columns = [c.lower() for c in d.columns]           # bulk rename
d = d.drop(columns=["bonus", "tax"])
d = d.drop(index=[0, 1])                             # drop rows by label
print(d)
print(d.pop("level").head(2))                        # remove & return a column

  employee   id  age    city     ctc     dept   level      net
2   Chitra  102   28    Pune   61000       IT  Normal  48800.0
3      Dev  103   45  Mumbai  120000  Finance    High  84000.0
4     Esha  104   38    Pune   98000       IT    High  68600.0
5    Farid  105   29   Delhi   58000       HR  Normal  46400.0
2    Normal
3      High
Name: level, dtype: str


### 🎯 Interview Points
- `drop` returns a **new** DataFrame unless `inplace=True` (inplace is discouraged; doesn't save memory generally).
- `axis=0` → rows, `axis=1` → columns. `drop(columns=...)` is clearer than `axis=1`.
- Use `assign` + method chaining for readable pipelines.
- `df.copy()` for independence; plain `df2 = df` makes just another reference.

## 6. Data Types & Conversion

In [18]:
t = pd.DataFrame({"n": ["1", "2", "x"], "d": ["2024-01-05", "2024-02-10", "bad"], "c": ["a", "b", "a"], "f": ["yes", "no", "yes"]})
print(t.dtypes)
t["n_num"] = pd.to_numeric(t["n"], errors="coerce")        # invalid -> NaN
t["d_dt"] = pd.to_datetime(t["d"], errors="coerce")        # invalid -> NaT
t["c"] = t["c"].astype("category")
t["f_bool"] = t["f"].map({"yes": True, "no": False})
print(t); print(t.dtypes)

n    str
d    str
c    str
f    str
dtype: object
   n           d  c    f  n_num       d_dt  f_bool
0  1  2024-01-05  a  yes    1.0 2024-01-05    True
1  2  2024-02-10  b   no    2.0 2024-02-10   False
2  x         bad  a  yes    NaN        NaT    True
n                    str
d                    str
c               category
f                    str
n_num            float64
d_dt      datetime64[us]
f_bool              bool
dtype: object


In [19]:
# Nullable dtypes (support missing values without turning ints into floats)
ints = pd.Series([1, 2, None])
print(ints.dtype)                          # float64 (NaN forces float)
print(ints.astype("Int64"))                # nullable integer with <NA>
print(pd.Series(["a", None], dtype="string").dtype)
# Categorical
cat = pd.Series(["low", "high", "mid", "low"], dtype=pd.CategoricalDtype(["low", "mid", "high"], ordered=True))
print(cat.sort_values().tolist(), cat.cat.codes.tolist(), cat.max())

float64
0       1
1       2
2    <NA>
dtype: Int64
string
['low', 'low', 'mid', 'high'] [0, 2, 1, 0] high


### 🎯 Interview Points
- `astype` raises on bad values; `pd.to_numeric(..., errors="coerce")` converts bad → NaN.
- Integer columns with `NaN` become **float64** (classic) → use nullable `Int64` (capital I).
- `category` dtype saves lots of memory for low-cardinality text and enables custom sort order.
- `object` dtype = Python objects (usually strings); in pandas 3.x strings get a dedicated `str` dtype.

## 7. Missing Data ⭐

In [20]:
m = pd.DataFrame({"a": [1, np.nan, 3, np.nan], "b": ["x", None, "z", "w"], "c": [10, 20, np.nan, 40]})
print(m.isna()); print(m.isna().sum()); print((m.isna().mean() * 100).round(1))

       a      b      c
0  False  False  False
1   True   True  False
2  False  False   True
3   True  False  False
a    2
b    1
c    1
dtype: int64
a    50.0
b    25.0
c    25.0
dtype: float64


In [21]:
print(m.dropna())                          # drop rows with ANY NaN
print(m.dropna(how="all"))                 # only rows where ALL NaN
print(m.dropna(subset=["a"]))              # only look at column a
print(m.dropna(thresh=2))                  # keep rows with >= 2 non-null values

     a  b     c
0  1.0  x  10.0
     a    b     c
0  1.0    x  10.0
1  NaN  NaN  20.0
2  3.0    z   NaN
3  NaN    w  40.0
     a  b     c
0  1.0  x  10.0
2  3.0  z   NaN
     a  b     c
0  1.0  x  10.0
2  3.0  z   NaN
3  NaN  w  40.0


In [22]:
print(m.fillna({"a": m["a"].mean(), "b": "unknown", "c": 0}))
print(m["a"].ffill(), m["c"].bfill())      # forward/back fill (time series)
print(m["c"].interpolate())                # linear interpolation
print(m.replace({"x": "X"}))
# Group-wise fill
g = pd.DataFrame({"k": list("AABB"), "v": [1, np.nan, 5, np.nan]})
g["v"] = g.groupby("k")["v"].transform(lambda s: s.fillna(s.mean()))
print(g)

     a        b     c
0  1.0        x  10.0
1  2.0  unknown  20.0
2  3.0        z   0.0
3  2.0        w  40.0
0    1.0
1    1.0
2    3.0
3    3.0
Name: a, dtype: float64 0    10.0
1    20.0
2    40.0
3    40.0
Name: c, dtype: float64
0    10.0
1    20.0
2    30.0
3    40.0
Name: c, dtype: float64
     a    b     c
0  1.0    X  10.0
1  NaN  NaN  20.0
2  3.0    z   NaN
3  NaN    w  40.0
   k    v
0  A  1.0
1  A  1.0
2  B  5.0
3  B  5.0


### 🎯 Interview Points
- Missing markers: `np.nan`, `None`, `pd.NaT` (datetime), `pd.NA` (nullable dtypes). `isna()` detects all.
- `NaN != NaN`; use `isna()` not `== np.nan`.
- Aggregations (`sum`, `mean`) **skip NaN by default** (`skipna=True`) — unlike NumPy.
- Strategies: drop (`dropna`), constant/statistic fill, ffill/bfill, interpolate, group-wise fill, model-based imputation.
- Report **percentage missing** before deciding; dropping may bias data.

## 8. Duplicates

In [23]:
dup = pd.DataFrame({"id": [1, 2, 2, 3, 3], "v": ["a", "b", "b", "c", "d"]})
print(dup.duplicated())                     # marks 2nd+ occurrences as True
print(dup.duplicated(keep=False))           # marks ALL duplicates
print(dup.drop_duplicates())
print(dup.drop_duplicates(subset="id", keep="last"))

0    False
1    False
2     True
3    False
4    False
dtype: bool
0    False
1     True
2     True
3    False
4    False
dtype: bool
   id  v
0   1  a
1   2  b
3   3  c
4   3  d
   id  v
0   1  a
2   2  b
4   3  d


## 9. String Operations (`.str`)

In [24]:
s = pd.Series(["  Alice Smith ", "bob JONES", "Charlie-Brown", None, "dave@mail.com"])
print(s.str.strip().str.title())
print(s.str.lower().str.contains("o", na=False))
print(s.str.len().tolist())
print(s.str.replace(r"[^A-Za-z ]", "", regex=True))
print(s.str.split(" ", expand=True))
print(s.str.extract(r"(\w+)@(\w+)\.com"))
print(s.str.startswith("bob").tolist())
print(pd.Series(["a,b", "c"]).str.split(",").explode().tolist())
print(pd.Series(["x", "y"]).str.cat(sep="-"))

0      Alice Smith
1        Bob Jones
2    Charlie-Brown
3              NaN
4    Dave@Mail.Com
dtype: str
0    False
1     True
2     True
3    False
4     True
dtype: bool
[14.0, 9.0, 13.0, nan, 13.0]
0      Alice Smith 
1         bob JONES
2      CharlieBrown
3               NaN
4       davemailcom
dtype: str
               0      1      2      3    4
0                        Alice  Smith     
1            bob  JONES    NaN    NaN  NaN
2  Charlie-Brown    NaN    NaN    NaN  NaN
3            NaN    NaN    NaN    NaN  NaN
4  dave@mail.com    NaN    NaN    NaN  NaN
      0     1
0   NaN   NaN
1   NaN   NaN
2   NaN   NaN
3   NaN   NaN
4  dave  mail
[False, True, False, False, False]
['a', 'b', 'c']
x-y


### 🎯 Interview Points
- `.str` methods are vectorized & **NaN-safe** (return NaN for missing).
- `str.contains(..., regex=True)` by default; pass `na=False` when using as a mask.
- `expand=True` in `split` returns a DataFrame; `explode` converts list items into rows.

## 10. Date & Time ⭐

In [25]:
dates = pd.DataFrame({"date": pd.date_range("2024-01-29", periods=8, freq="D"), "sales": [5, 7, 3, 8, 6, 9, 4, 10]})
dates["year"] = dates["date"].dt.year
dates["month"] = dates["date"].dt.month
dates["day_name"] = dates["date"].dt.day_name()
dates["is_weekend"] = dates["date"].dt.dayofweek >= 5
dates["month_end"] = dates["date"] + pd.offsets.MonthEnd(0)
print(dates)

        date  sales  year  month   day_name  is_weekend  month_end
0 2024-01-29      5  2024      1     Monday       False 2024-01-31
1 2024-01-30      7  2024      1    Tuesday       False 2024-01-31
2 2024-01-31      3  2024      1  Wednesday       False 2024-01-31
3 2024-02-01      8  2024      2   Thursday       False 2024-02-29
4 2024-02-02      6  2024      2     Friday       False 2024-02-29
5 2024-02-03      9  2024      2   Saturday        True 2024-02-29
6 2024-02-04      4  2024      2     Sunday        True 2024-02-29
7 2024-02-05     10  2024      2     Monday       False 2024-02-29


In [26]:
ts = dates.set_index("date")["sales"]
print(ts["2024-02-01":"2024-02-03"])                 # slicing by date strings
print(ts.resample("W").sum())                        # weekly total
print(ts.resample("2D").mean())
print(pd.Timestamp("2024-03-01") - pd.Timestamp("2024-01-01"))   # Timedelta
print((pd.to_datetime("2024-03-01") - pd.to_datetime("2024-01-01")).days)
print(pd.to_datetime("05/12/2024", format="%d/%m/%Y"))
print(ts.index.to_period("M")[:2])

date
2024-02-01    8
2024-02-02    6
2024-02-03    9
Name: sales, dtype: int64
date
2024-02-04    42
2024-02-11    10
Freq: W-SUN, Name: sales, dtype: int64
date
2024-01-29    6.0
2024-01-31    5.5
2024-02-02    7.5
2024-02-04    7.0
Freq: 2D, Name: sales, dtype: float64
60 days 00:00:00
60
2024-12-05 00:00:00
PeriodIndex(['2024-01', '2024-01'], dtype='period[M]', name='date')


### 🎯 Interview Points
- Convert text → datetime with `pd.to_datetime` (use `format=` for speed & to avoid day/month ambiguity).
- `.dt` accessor for datetime Series; DatetimeIndex enables **partial string slicing** and `resample`.
- `resample` = time-based groupby (needs DatetimeIndex or `on="col"`). Month-end alias is `"ME"` (older versions: `"M"`).
- `Timestamp`, `Timedelta`, `Period`, `DateOffset` are the 4 core time types.

## 11. Sorting & Ranking

In [27]:
print(df.sort_values("salary", ascending=False).head(3))
print(df.sort_values(["dept", "salary"], ascending=[True, False])[["dept", "name", "salary"]])
print(df.sort_index().head(2))
print(df.nlargest(2, "salary")[["name", "salary"]], df.nsmallest(2, "age")[["name", "age"]], sep="\n")
df["rank"] = df["salary"].rank(ascending=False, method="dense").astype(int)
df["rank_in_dept"] = df.groupby("dept")["salary"].rank(ascending=False)
print(df[["name", "dept", "salary", "rank", "rank_in_dept"]])

   name  age    city  salary     dept level
3   Dev   45  Mumbai  120000  Finance  High
4  Esha   38    Pune   98000       IT  High
1   Ben   32   Delhi   72000       HR  High
      dept    name  salary
3  Finance     Dev  120000
1       HR     Ben   72000
5       HR   Farid   58000
4       IT    Esha   98000
2       IT  Chitra   61000
0       IT    Asha   50000
   name  age    city  salary dept   level
0  Asha   25  Mumbai   50000   IT  Normal
1   Ben   32   Delhi   72000   HR    High
   name  salary
3   Dev  120000
4  Esha   98000
     name  age
0    Asha   25
2  Chitra   28
     name     dept  salary  rank  rank_in_dept
0    Asha       IT   50000     6           3.0
1     Ben       HR   72000     3           1.0
2  Chitra       IT   61000     4           2.0
3     Dev  Finance  120000     1           1.0
4    Esha       IT   98000     2           1.0
5   Farid       HR   58000     5           2.0


### 🎯 Interview Points
- `rank` methods: `average` (default), `min`, `max`, `first`, `dense`.
- `nlargest`/`nsmallest` are faster than sort + head.
- `sort_values` isn't stable by default except with `kind="stable"` / mergesort; matters for multi-key ties.

## 12. `apply`, `map`, `applymap` & Vectorization ⭐

In [28]:
x = df[["name", "salary"]].copy()
x["grade"] = x["salary"].map(lambda v: "A" if v > 90000 else "B" if v > 60000 else "C")   # Series.map
x["dept_code"] = df["dept"].map({"IT": 1, "HR": 2, "Finance": 3})                          # map via dict
x["name_len"] = x["name"].apply(len)                                                       # Series.apply
print(x)

# DataFrame.apply on rows (axis=1) - slow, use only if needed
row_sum = df[["age", "salary"]].apply(lambda r: r["age"] * 1000 + r["salary"], axis=1)
# DataFrame.apply on columns (axis=0)
print(df[["age", "salary"]].apply(lambda c: c.max() - c.min()))
# Element-wise on whole DataFrame (applymap was renamed to DataFrame.map)
print(df[["age", "salary"]].map(lambda v: v / 1000))

     name  salary grade  dept_code  name_len
0    Asha   50000     C          1         4
1     Ben   72000     B          2         3
2  Chitra   61000     B          1         6
3     Dev  120000     A          3         3
4    Esha   98000     A          1         4
5   Farid   58000     C          2         5
age          20
salary    70000
dtype: int64
     age  salary
0  0.025    50.0
1  0.032    72.0
2  0.028    61.0
3  0.045   120.0
4  0.038    98.0
5  0.029    58.0


In [29]:
import time
big = pd.DataFrame({"a": np.arange(200_000), "b": np.arange(200_000) * 2})
t0 = time.perf_counter(); r1 = big.apply(lambda r: r["a"] + r["b"], axis=1); t_apply = time.perf_counter() - t0
t0 = time.perf_counter(); r2 = big["a"] + big["b"]; t_vec = time.perf_counter() - t0
print(f"apply(axis=1): {t_apply:.3f}s | vectorized: {t_vec*1000:.2f}ms | {t_apply/t_vec:.0f}x slower")

apply(axis=1): 1.541s | vectorized: 1.23ms | 1257x slower


### 🎯 Interview Points
- Speed order: **vectorized ops** > `np.where`/`np.select` > `map`/`str` > `apply(axis=1)` > `iterrows` (slowest).
- `Series.map` (element-wise, accepts dict/Series/function), `Series.apply` (function), `DataFrame.apply` (per row/col), `DataFrame.map` (element-wise; formerly `applymap`).
- Avoid `iterrows()`; if needed use `itertuples()` (much faster).
- Multi-condition → `np.select(conditions, choices, default)`.

## 13. GroupBy — Split → Apply → Combine ⭐⭐

In [30]:
print(df.groupby("dept")["salary"].mean())
print(df.groupby("dept")["salary"].agg(["count", "mean", "max"]))
print(df.groupby(["dept", "city"])["salary"].sum())
print(df.groupby("dept").size())

dept
Finance    120000.000000
HR          65000.000000
IT          69666.666667
Name: salary, dtype: float64
         count           mean     max
dept                                 
Finance      1  120000.000000  120000
HR           2   65000.000000   72000
IT           3   69666.666667   98000
dept     city  
Finance  Mumbai    120000
HR       Delhi     130000
IT       Mumbai     50000
         Pune      159000
Name: salary, dtype: int64
dept
Finance    1
HR         2
IT         3
dtype: int64


In [31]:
# Named aggregation (clean output names) - interview favourite
res = df.groupby("dept").agg(
    headcount=("name", "count"),
    avg_salary=("salary", "mean"),
    oldest=("age", "max"),
    total=("salary", "sum"),
)
print(res)
print(df.groupby("dept").agg({"salary": ["min", "max"], "age": "mean"}))

         headcount     avg_salary  oldest   total
dept                                             
Finance          1  120000.000000      45  120000
HR               2   65000.000000      32  130000
IT               3   69666.666667      38  209000
         salary                age
            min     max       mean
dept                              
Finance  120000  120000  45.000000
HR        58000   72000  30.500000
IT        50000   98000  30.333333


In [32]:
# transform: returns same-length result (broadcast back to rows)
df["dept_avg"] = df.groupby("dept")["salary"].transform("mean")
df["pct_of_dept"] = (df["salary"] / df.groupby("dept")["salary"].transform("sum") * 100).round(1)
print(df[["name", "dept", "salary", "dept_avg", "pct_of_dept"]])

# filter: keep whole groups meeting a condition
print(df.groupby("dept").filter(lambda g: g["salary"].mean() > 70000)[["name", "dept"]])

# Top-N per group
print(df.sort_values("salary", ascending=False).groupby("dept").head(1)[["dept", "name", "salary"]])

# Iterating over groups
for name, grp in df.groupby("city"):
    print(name, len(grp), end=" | ")

     name     dept  salary       dept_avg  pct_of_dept
0    Asha       IT   50000   69666.666667         23.9
1     Ben       HR   72000   65000.000000         55.4
2  Chitra       IT   61000   69666.666667         29.2
3     Dev  Finance  120000  120000.000000        100.0
4    Esha       IT   98000   69666.666667         46.9
5   Farid       HR   58000   65000.000000         44.6
  name     dept
3  Dev  Finance
      dept  name  salary
3  Finance   Dev  120000
4       IT  Esha   98000
1       HR   Ben   72000
Delhi 2 | Mumbai 2 | Pune 2 | 

### 🎯 Interview Points
- **`agg`** → reduces each group to one row. **`transform`** → returns same shape as input (e.g. z-score within group, group mean per row). **`filter`** → keeps/drops whole groups. **`apply`** → flexible, slowest.
- `groupby` ignores NaN keys by default (`dropna=False` to keep).
- `as_index=False` (or `reset_index()`) returns group keys as columns.
- `size()` counts rows incl. NaN; `count()` counts non-null per column.
- `groupby` + `.agg` with **named aggregation** gives clean column names.
- Mental model: *split by key → apply function independently → combine results*.

## 14. Reshaping: pivot, pivot_table, crosstab, melt, stack/unstack

In [33]:
sales = pd.DataFrame({
    "month": ["Jan", "Jan", "Feb", "Feb", "Mar", "Mar"],
    "region": ["N", "S", "N", "S", "N", "S"],
    "revenue": [100, 150, 120, 130, 90, 160],
    "units": [10, 15, 12, 13, 9, 16],
})
print(sales.pivot(index="month", columns="region", values="revenue"))            # long -> wide (no duplicates allowed)
print(sales.pivot_table(index="region", values=["revenue", "units"], aggfunc="sum", margins=True))
print(pd.pivot_table(df, index="dept", columns="city", values="salary", aggfunc="mean", fill_value=0))
print(pd.crosstab(df["dept"], df["city"]))
print(pd.crosstab(df["dept"], df["city"], normalize="index").round(2))

region    N    S
month           
Feb     120  130
Jan     100  150
Mar      90  160
        revenue  units
region                
N           310     31
S           440     44
All         750     75
city       Delhi    Mumbai     Pune
dept                               
Finance      0.0  120000.0      0.0
HR       65000.0       0.0      0.0
IT           0.0   50000.0  79500.0
city     Delhi  Mumbai  Pune
dept                        
Finance      0       1     0
HR           2       0     0
IT           0       1     2
city     Delhi  Mumbai  Pune
dept                        
Finance    0.0    1.00  0.00
HR         1.0    0.00  0.00
IT         0.0    0.33  0.67


In [34]:
# melt: wide -> long
wide = pd.DataFrame({"id": [1, 2], "math": [90, 80], "sci": [70, 85]})
long = wide.melt(id_vars="id", var_name="subject", value_name="score")
print(long)
print(long.pivot(index="id", columns="subject", values="score"))      # back to wide

# stack / unstack work with the index
stk = wide.set_index("id").stack()
print(stk); print(stk.unstack())

   id subject  score
0   1    math     90
1   2    math     80
2   1     sci     70
3   2     sci     85
subject  math  sci
id                
1          90   70
2          80   85
id      
1   math    90
    sci     70
2   math    80
    sci     85
dtype: int64
    math  sci
id           
1     90   70
2     80   85


### 🎯 Interview Points
- `pivot` → pure reshape, **errors on duplicate index/column pairs**. `pivot_table` → aggregates duplicates (default `mean`).
- `melt` = unpivot (wide → long); `pivot` = long → wide. Long format is ideal for plotting (seaborn) & groupby.
- `crosstab` = frequency table; `normalize=` for proportions.
- `stack` moves columns → index level; `unstack` the reverse.

## 15. Combining Data: `merge`, `join`, `concat` ⭐

In [35]:
emp = pd.DataFrame({"emp_id": [1, 2, 3, 4], "name": ["A", "B", "C", "D"], "dept_id": [10, 20, 10, 40]})
dpt = pd.DataFrame({"dept_id": [10, 20, 30], "dept_name": ["IT", "HR", "Ops"]})

print("inner:\n", emp.merge(dpt, on="dept_id", how="inner"))
print("left:\n", emp.merge(dpt, on="dept_id", how="left"))
print("right:\n", emp.merge(dpt, on="dept_id", how="right"))
print("outer:\n", emp.merge(dpt, on="dept_id", how="outer", indicator=True))

inner:
    emp_id name  dept_id dept_name
0       1    A       10        IT
1       2    B       20        HR
2       3    C       10        IT
left:
 

   emp_id name  dept_id dept_name
0       1    A       10        IT
1       2    B       20        HR
2       3    C       10        IT
3       4    D       40       NaN
right:
    emp_id name  dept_id dept_name
0     1.0    A       10        IT
1     3.0    C       10        IT
2     2.0    B       20        HR
3     NaN  NaN       30       Ops
outer:
    emp_id name  dept_id dept_name      _merge
0     1.0    A       10        IT        both
1     3.0    C       10        IT        both
2     2.0    B       20        HR        both
3     NaN  NaN       30       Ops  right_only
4     4.0    D       40       NaN   left_only


In [36]:
# Different key names, suffixes, validation
a = pd.DataFrame({"k1": [1, 2], "v": ["a", "b"]})
b = pd.DataFrame({"k2": [1, 2], "v": ["x", "y"]})
print(a.merge(b, left_on="k1", right_on="k2", suffixes=("_left", "_right")))
try:
    pd.DataFrame({"k": [1, 1]}).merge(pd.DataFrame({"k": [1, 1], "z": [1, 2]}), on="k", validate="one_to_one")
except Exception as e:
    print(type(e).__name__, "-", str(e)[:70])

# Anti-join: rows in emp with no matching dept
m = emp.merge(dpt, on="dept_id", how="left", indicator=True)
print(m[m["_merge"] == "left_only"][["emp_id", "name"]])

# join uses the index
print(emp.set_index("dept_id").join(dpt.set_index("dept_id"), how="inner"))

   k1 v_left  k2 v_right
0   1      a   1       x
1   2      b   2       y


MergeError - Merge keys are not unique in either left or right dataset; not a one-t
   emp_id name
3       4    D
         emp_id name dept_name
dept_id                       
10            1    A        IT
20            2    B        HR
10            3    C        IT


In [37]:
# concat
d1 = pd.DataFrame({"a": [1, 2], "b": [3, 4]})
d2 = pd.DataFrame({"a": [5, 6], "c": [7, 8]})
print(pd.concat([d1, d2], ignore_index=True))               # stack rows (union of columns)
print(pd.concat([d1, d2], axis=1))                          # side by side (index aligned)
print(pd.concat([d1, d2], join="inner", ignore_index=True)) # only common columns
print(pd.concat([d1, d2], keys=["first", "second"]))        # hierarchical index

   a    b    c
0  1  3.0  NaN
1  2  4.0  NaN
2  5  NaN  7.0
3  6  NaN  8.0
   a  b  a  c
0  1  3  5  7
1  2  4  6  8
   a
0  1
1  2
2  5
3  6
          a    b    c
first  0  1  3.0  NaN
       1  2  4.0  NaN
second 0  5  NaN  7.0
       1  6  NaN  8.0


### 🎯 Interview Points
- `merge` types: **inner** (matches only), **left** (all left), **right**, **outer** (all) — same as SQL joins. Also `cross`.
- `merge` joins on **columns**; `join` joins on **index**; `concat` **stacks** along an axis.
- Duplicate keys → **row explosion** (many-to-many). Guard with `validate="one_to_one" / "one_to_many"`.
- `indicator=True` adds `_merge` column → perfect for anti-joins & debugging.
- `pd.merge_asof` joins to the nearest key (time-series).
- `DataFrame.append` was **removed** → use `concat`. Never `concat` inside a loop; collect in a list then concat once.

## 16. Window Functions, `shift`, `diff`, `pct_change`

In [38]:
px = pd.DataFrame({"day": pd.date_range("2024-01-01", periods=8),
                   "price": [100, 102, 101, 105, 110, 108, 112, 115]}).set_index("day")
px["prev"] = px["price"].shift(1)
px["change"] = px["price"].diff()
px["pct"] = (px["price"].pct_change() * 100).round(2)
px["ma3"] = px["price"].rolling(3).mean()
px["ma3_min2"] = px["price"].rolling(3, min_periods=2).mean()
px["cummax"] = px["price"].cummax()
px["expanding_avg"] = px["price"].expanding().mean().round(2)
px["ewm"] = px["price"].ewm(span=3).mean().round(2)
print(px)

            price   prev  change   pct         ma3    ma3_min2  cummax  \
day                                                                      
2024-01-01    100    NaN     NaN   NaN         NaN         NaN     100   
2024-01-02    102  100.0     2.0  2.00         NaN  101.000000     102   
2024-01-03    101  102.0    -1.0 -0.98  101.000000  101.000000     102   
2024-01-04    105  101.0     4.0  3.96  102.666667  102.666667     105   
2024-01-05    110  105.0     5.0  4.76  105.333333  105.333333     110   
2024-01-06    108  110.0    -2.0 -1.82  107.666667  107.666667     110   
2024-01-07    112  108.0     4.0  3.70  110.000000  110.000000     112   
2024-01-08    115  112.0     3.0  2.68  111.666667  111.666667     115   

            expanding_avg     ewm  
day                                
2024-01-01         100.00  100.00  
2024-01-02         101.00  101.33  
2024-01-03         101.00  101.14  
2024-01-04         102.00  103.20  
2024-01-05         103.60  106.71  
2024-01

In [39]:
# Group-wise shift / cumulative sum
g = pd.DataFrame({"k": list("AAABBB"), "v": [1, 2, 3, 10, 20, 30]})
g["prev_in_group"] = g.groupby("k")["v"].shift()
g["cum"] = g.groupby("k")["v"].cumsum()
print(g)

   k   v  prev_in_group  cum
0  A   1            NaN    1
1  A   2            1.0    3
2  A   3            2.0    6
3  B  10            NaN   10
4  B  20           10.0   30
5  B  30           20.0   60


### 🎯 Interview Points
- `rolling(window)` fixed-size sliding; `expanding()` grows from the start; `ewm` exponentially weights recent data.
- First `window-1` rows of rolling are NaN unless `min_periods` is set.
- `shift(n)` for lag features (ML), `diff` for differences, `pct_change` for returns.
- These can be combined with `groupby` for per-group window calculations.

## 17. MultiIndex (Hierarchical Index)

In [40]:
mi = df.groupby(["dept", "city"])["salary"].mean()
print(mi)
print(mi.loc["IT"])
print(mi.loc[("IT", "Pune")])
print(mi.unstack())                       # inner level -> columns
print(mi.reset_index())
print(mi.xs("Pune", level="city"))        # cross-section on inner level

dept     city  
Finance  Mumbai    120000.0
HR       Delhi      65000.0
IT       Mumbai     50000.0
         Pune       79500.0
Name: salary, dtype: float64
city
Mumbai    50000.0
Pune      79500.0
Name: salary, dtype: float64
79500.0
city       Delhi    Mumbai     Pune
dept                               
Finance      NaN  120000.0      NaN
HR       65000.0       NaN      NaN
IT           NaN   50000.0  79500.0
      dept    city    salary
0  Finance  Mumbai  120000.0
1       HR   Delhi   65000.0
2       IT  Mumbai   50000.0
3       IT    Pune   79500.0
dept
IT    79500.0
Name: salary, dtype: float64


### 🎯 Interview Points
- Created by `set_index([a, b])` or multi-key `groupby`. Flatten using `reset_index()` or `df.columns = ["_".join(c) for c in df.columns]`.
- `xs`, `loc` with tuples, and `unstack(level=)` navigate levels. Sort the index (`sort_index()`) for fast slicing.

## 18. Binning & Counting

In [41]:
ages = df["age"]
print(pd.cut(ages, bins=[0, 30, 40, 100], labels=["<=30", "31-40", "40+"]))
print(pd.qcut(df["salary"], q=3, labels=["Low", "Mid", "High"]))     # equal-sized quantile buckets
print(df["city"].value_counts())
print(df["city"].value_counts(normalize=True).round(2))
print(df["salary"].quantile([0.25, 0.5, 0.75]))
print(df[["age", "salary"]].corr().round(2))
print(df["salary"].agg(["mean", "median", "std", "skew"]).round(2))

0     <=30
1    31-40
2     <=30
3      40+
4    31-40
5     <=30
Name: age, dtype: category
Categories (3, str): ['<=30' < '31-40' < '40+']
0     Low
1     Mid
2     Mid
3    High
4    High
5     Low
Name: salary, dtype: category
Categories (3, str): ['Low' < 'Mid' < 'High']
city
Mumbai    2
Delhi     2
Pune      2
Name: count, dtype: int64
city
Mumbai    0.33
Delhi     0.33
Pune      0.33
Name: proportion, dtype: float64
0.25    58750.0
0.50    66500.0
0.75    91500.0
Name: salary, dtype: float64
         age  salary
age     1.00    0.99
salary  0.99    1.00
mean      76500.00
median    66500.00
std       27053.65
skew          0.96
Name: salary, dtype: float64


### 🎯 Interview Points
- `cut` = **fixed-width/custom bins**; `qcut` = **equal-frequency** (quantile) bins.
- `value_counts()` sorts descending, excludes NaN (`dropna=False` to include).
- `corr()` default Pearson; `method="spearman"` for rank correlation.
- Outlier rule (IQR): `Q1 - 1.5*IQR` , `Q3 + 1.5*IQR`.

In [42]:
# IQR outlier detection
q1, q3 = df["salary"].quantile([.25, .75]); iqr = q3 - q1
print(df[(df["salary"] < q1 - 1.5 * iqr) | (df["salary"] > q3 + 1.5 * iqr)])
print(df["salary"].clip(lower=55000, upper=100000).tolist())      # winsorize / cap

Empty DataFrame
Columns: [name, age, city, salary, dept, level, rank, rank_in_dept, dept_avg, pct_of_dept]
Index: []
[55000, 72000, 61000, 100000, 98000, 58000]


## 19. Reading & Writing Files

In [43]:
import os, tempfile, io
tmp = tempfile.mkdtemp()
p = os.path.join(tmp, "emp.csv")
df[["name", "age", "city", "salary"]].to_csv(p, index=False)
print(open(p).read()[:80])

back = pd.read_csv(p, usecols=["name", "salary"], dtype={"name": "string"}, nrows=4)
print(back)

# Large files: chunked reading
tot = 0
for chunk in pd.read_csv(p, chunksize=2):
    tot += chunk["salary"].sum()
print("chunked total:", tot)

# JSON, string buffers, parse dates
print(df[["name", "age"]].head(2).to_json(orient="records"))
print(pd.read_csv(io.StringIO("d,v\n2024-01-01,1\n2024-01-02,2"), parse_dates=["d"]).dtypes)
# Other formats: to_excel (openpyxl), to_parquet (pyarrow), to_sql, read_sql, read_html, read_clipboard

name,age,city,salary
Asha,25,Mumbai,50000
Ben,32,Delhi,72000
Chitra,28,Pune,6100
     name  salary
0    Asha   50000
1     Ben   72000
2  Chitra   61000
3     Dev  120000
chunked total: 459000
[{"name":"Asha","age":25},{"name":"Ben","age":32}]
d    datetime64[us]
v             int64
dtype: object


### 🎯 Interview Points
- `read_csv` key params: `sep`, `header`, `names`, `usecols`, `dtype`, `parse_dates`, `na_values`, `nrows`, `skiprows`, `chunksize`, `encoding`.
- `to_csv(index=False)` to avoid writing the index column.
- **Parquet** is faster, smaller & preserves dtypes → better than CSV for big data.
- Millions of rows: `chunksize`, `usecols`, smaller `dtype`s, or move to Dask/Polars/SQL.

## 20. Performance & Memory Optimization

In [44]:
n = 300_000
rng = np.random.default_rng(0)
big = pd.DataFrame({
    "city": rng.choice(["Mumbai", "Delhi", "Pune", "Chennai"], n),
    "qty": rng.integers(0, 100, n),
    "price": rng.random(n) * 100,
})
before = big.memory_usage(deep=True).sum() / 1e6
opt = big.astype({"city": "category", "qty": "int8", "price": "float32"})
after = opt.memory_usage(deep=True).sum() / 1e6
print(f"memory: {before:.1f} MB -> {after:.1f} MB")
print(big.dtypes.to_dict(), opt.dtypes.to_dict())

# Fast vs slow
import time
t0 = time.perf_counter(); s1 = sum(r.qty * r.price for r in big.head(50000).itertuples()); t_it = time.perf_counter() - t0
t0 = time.perf_counter(); s2 = (big["qty"].head(50000) * big["price"].head(50000)).sum(); t_vec = time.perf_counter() - t0
print(f"itertuples {t_it*1000:.0f}ms vs vectorized {t_vec*1000:.2f}ms")

memory: 8.8 MB -> 1.8 MB
{'city': <StringDtype(na_value=nan)>, 'qty': dtype('int64'), 'price': dtype('float64')} {'city': CategoricalDtype(categories=['Chennai', 'Delhi', 'Mumbai', 'Pune'], ordered=False, categories_dtype=str), 'qty': dtype('int8'), 'price': dtype('float32')}
itertuples 104ms vs vectorized 1.37ms


### 🎯 Interview Points – Optimization checklist
1. Use **vectorized** ops, avoid `apply(axis=1)`/`iterrows`.
2. **Downcast dtypes** (`int8`, `float32`) and use `category` for repeated strings.
3. Load only needed columns (`usecols`) & use `chunksize`.
4. Use `query`/`eval` for big expressions; use `.to_numpy()` for heavy numeric work.
5. Store in **Parquet/Feather**, not CSV.
6. Avoid repeated `concat`/`append` in loops.
7. For out-of-memory data → Dask, Polars, DuckDB, PySpark.

## 21. Pitfalls ⚠️ (SettingWithCopy, Copy-on-Write, chaining)

In [45]:
a = pd.DataFrame({"x": [1, 2, 3], "y": [10, 20, 30]})

# BAD: chained assignment (unreliable; in pandas 3.0 with Copy-on-Write it NEVER modifies `a`)
sub = a[a["x"] > 1]
sub["y"] = 0                 # modifies a *copy* — original unchanged
print(a)

# GOOD: single .loc assignment
a.loc[a["x"] > 1, "y"] = 0
print(a)

# Explicit copy when you want an independent frame
b = a.copy(); b.loc[0, "x"] = 999; print(a["x"].tolist(), b["x"].tolist())

   x   y
0  1  10
1  2  20
2  3  30
   x   y
0  1  10
1  2   0
2  3   0
[1, 2, 3] [999, 2, 3]


In [46]:
# Other classic gotchas
# 1) Integer + NaN -> float
print(pd.Series([1, 2, np.nan]).dtype)
# 2) Comparing with NaN
print(pd.Series([1, np.nan]) == np.nan)
# 3) Index alignment surprise
print(pd.Series([1, 2], index=[0, 1]) + pd.Series([10, 20], index=[1, 2]))
# 4) Boolean operators need &, |, ~ and parentheses
# 5) Merge duplicates explode rows
# 6) After filtering, index has gaps: use reset_index(drop=True)
# 7) inplace=True does not return anything (returns None)
print(a.drop(columns="y", inplace=True))

float64
0    False
1    False
dtype: bool
0     NaN
1    12.0
2     NaN
dtype: float64
None


### 🎯 Interview Points
- **SettingWithCopyWarning** arises from *chained assignment* (`df[mask]["col"] = v`). Fix: `df.loc[mask, "col"] = v`, or `.copy()` when slicing intentionally.
- **Copy-on-Write (CoW)**: in pandas 3.0 it's the default — any derived object behaves like a copy; modifying it never changes the parent. Makes behaviour predictable and removes the warning.
- `inplace=True` returns `None` → `df = df.dropna(inplace=True)` makes `df = None` 🙃.

## 22. Mini Project: End-to-End Sales Analysis 🛒
Combines cleaning → feature engineering → groupby → pivot → time series.

In [47]:
rng = np.random.default_rng(7)
n = 500
orders = pd.DataFrame({
    "order_id": np.arange(1, n + 1),
    "date": pd.to_datetime("2024-01-01") + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
    "customer_id": rng.integers(1, 80, n),
    "category": rng.choice(["Electronics", "Clothing", "Grocery", "Books"], n, p=[.25, .3, .35, .1]),
    "qty": rng.integers(1, 6, n),
    "unit_price": rng.choice([99.0, 249.0, 15.5, 499.0, 35.0], n),
})
# introduce dirt
orders.loc[rng.choice(n, 15, replace=False), "unit_price"] = np.nan
orders = pd.concat([orders, orders.sample(10, random_state=1)], ignore_index=True)    # duplicates
print(orders.shape, "| nulls:", orders["unit_price"].isna().sum(), "| dups:", orders.duplicated().sum())

(510, 6) | nulls: 15 | dups: 10


In [48]:
# 1. Clean
clean = (orders
         .drop_duplicates()
         .assign(unit_price=lambda d: d["unit_price"].fillna(d.groupby("category")["unit_price"].transform("median")))
         .assign(revenue=lambda d: d["qty"] * d["unit_price"],
                 month=lambda d: d["date"].dt.to_period("M").astype(str)))
print(clean.shape, clean["unit_price"].isna().sum())

# 2. KPIs
print("Total revenue:", round(clean["revenue"].sum(), 2), "| AOV:", round(clean["revenue"].mean(), 2))

# 3. Revenue by category
print(clean.groupby("category").agg(orders=("order_id", "count"), revenue=("revenue", "sum"),
      avg_qty=("qty", "mean")).sort_values("revenue", ascending=False).round(2))

# 4. Month x category pivot
print(clean.pivot_table(index="month", columns="category", values="revenue", aggfunc="sum", fill_value=0).round(0))

# 5. Top 5 customers
top = clean.groupby("customer_id")["revenue"].sum().nlargest(5)
print(top)

# 6. Monthly growth
monthly = clean.groupby("month")["revenue"].sum()
print(pd.DataFrame({"revenue": monthly, "mom_growth_%": (monthly.pct_change() * 100).round(1)}))

# 7. Repeat customers
cnt = clean["customer_id"].value_counts()
print("Repeat customer rate:", round((cnt > 1).mean() * 100, 1), "%")

# 8. Weekly rolling revenue
weekly = clean.set_index("date")["revenue"].resample("W").sum()
print(weekly.rolling(4).mean().dropna().head())

(500, 8) 0
Total revenue: 307472.0 | AOV: 614.94
             orders  revenue  avg_qty
category                             
Electronics     135  98169.5     3.23
Clothing        154  95207.5     3.14
Grocery         162  91420.5     2.91
Books            49  22674.5     3.02
category   Books  Clothing  Electronics  Grocery
month                                           
2024-01   2720.0   15572.0      21062.0  12510.0
2024-02   4138.0   20277.0      17694.0  16513.0
2024-03   4835.0   11048.0      14586.0  14682.0
2024-04   4750.0   18053.0      20559.0  17800.0
2024-05   4129.0   12871.0      16208.0  12594.0
2024-06   2103.0   17386.0       8060.0  17322.0
customer_id
67    11203.0
46     8982.0
15     8891.0
30     8575.0
31     8225.5
Name: revenue, dtype: float64
         revenue  mom_growth_%
month                         
2024-01  51865.0           NaN
2024-02  58621.0          13.0
2024-03  45150.5         -23.0
2024-04  61161.0          35.5
2024-05  45802.5         -25.1
20

## 23. Interview Q&A (Rapid-fire) 🎤

**Basic**
1. **Series vs DataFrame?** 1-D labelled array vs 2-D labelled table.
2. **`loc` vs `iloc`?** Label (end inclusive) vs position (end exclusive).
3. **How to select multiple columns?** `df[["a", "b"]]`.
4. **How to see missing values?** `df.isna().sum()`.
5. **`head` vs `sample`?** First n rows vs random n rows.
6. **`df.shape` vs `df.size` vs `len(df)`?** (rows, cols) vs total cells vs rows.
7. **How do you rename columns?** `df.rename(columns={...})` or `df.columns = [...]`.
8. **`reset_index` vs `set_index`?** Move index → column vs column → index; `drop=True` discards.

**Intermediate**
9. **`merge` vs `join` vs `concat`?** Column-key join / index join / stack along axis.
10. **Inner / left / outer join?** Same as SQL; `how=` parameter.
11. **`apply` vs `map` vs `applymap`?** `map`: Series element-wise; `apply`: Series or per axis of DataFrame; `DataFrame.map` (old `applymap`): element-wise over whole DataFrame.
12. **`agg` vs `transform` in groupby?** `agg` reduces; `transform` keeps original shape.
13. **`pivot` vs `pivot_table`?** pivot can't aggregate duplicates; pivot_table aggregates.
14. **`melt` use?** Wide → long.
15. **How to handle missing data?** Drop / fill / ffill / interpolate / group-wise / model.
16. **How to find & remove duplicates?** `duplicated()`, `drop_duplicates(subset, keep)`.
17. **`cut` vs `qcut`?** Value-range bins vs equal-count bins.
18. **How to convert a column to datetime?** `pd.to_datetime(col, format=..., errors="coerce")`.
19. **`value_counts` vs `groupby().size()`?** Convenience for a single column vs general grouping.
20. **Why is `iterrows` slow?** Builds a Series per row; Python-level loop.

**Advanced**
21. **What is SettingWithCopyWarning?** Chained assignment ambiguity (view vs copy). Use `.loc` / `.copy()`. With CoW (default in 3.0) it's resolved.
22. **Index alignment?** Operations match labels, produce NaN where unmatched.
23. **How to optimise memory?** Downcast, `category`, `usecols`, chunking, Parquet.
24. **Explain `groupby` internally.** Split → apply → combine; hash-based grouping; lazy until aggregation.
25. **How to get top-N per group?** `sort_values().groupby().head(n)` or `groupby().nlargest`, or `rank()`.
26. **How to do a window function like SQL `ROW_NUMBER`?** `groupby().cumcount()` / `rank(method="first")`.
27. **How to do a self-join / anti-join?** `merge(..., indicator=True)` and filter `_merge`.
28. **Categorical dtype benefits?** Memory, speed, custom order.
29. **`copy()` vs view?** Pandas 3.0 Copy-on-Write: logically always a copy.
30. **Pandas vs SQL vs Polars?** Pandas: in-memory, flexible, huge ecosystem; SQL: set-based DB; Polars: faster, multi-threaded, lazy API.

## 24. Practice Problems (try first!) 🧠

In [49]:
emp = pd.DataFrame({
    "id": range(1, 9),
    "name": list("ABCDEFGH"),
    "dept": ["IT", "IT", "HR", "HR", "Fin", "Fin", "IT", "HR"],
    "salary": [90, 85, 60, 65, 100, 95, 70, 62],
    "joined": pd.to_datetime(["2020-01-10", "2019-03-15", "2021-07-01", "2018-11-20",
                              "2017-05-05", "2022-02-14", "2023-01-09", "2020-09-30"]),
})

# P1. Highest paid employee in each dept
print(emp.loc[emp.groupby("dept")["salary"].idxmax(), ["dept", "name", "salary"]])

# P2. 2nd highest salary overall
print(emp["salary"].drop_duplicates().nlargest(2).iloc[-1])

# P3. Employees earning more than their dept average
print(emp[emp["salary"] > emp.groupby("dept")["salary"].transform("mean")][["name", "dept", "salary"]])

# P4. Tenure in years (as of 2024-01-01) and bucket it
emp["tenure"] = ((pd.Timestamp("2024-01-01") - emp["joined"]).dt.days / 365.25).round(1)
emp["band"] = pd.cut(emp["tenure"], [0, 2, 5, 10], labels=["0-2", "2-5", "5+"])
print(emp[["name", "tenure", "band"]])

# P5. Department-wise salary share (%)
print((emp.groupby("dept")["salary"].sum() / emp["salary"].sum() * 100).round(1))

# P6. Running total of salary ordered by join date
print(emp.sort_values("joined").assign(cum=lambda d: d["salary"].cumsum())[["name", "joined", "cum"]].head(4))

# P7. Rank within dept (SQL: DENSE_RANK() OVER (PARTITION BY dept ORDER BY salary DESC))
emp["rk"] = emp.groupby("dept")["salary"].rank(method="dense", ascending=False).astype(int)
print(emp[["dept", "name", "salary", "rk"]].sort_values(["dept", "rk"]))

# P8. Count of employees who joined per year
print(emp["joined"].dt.year.value_counts().sort_index())

# P9. Departments having more than 2 employees
print(emp.groupby("dept").filter(lambda g: len(g) > 2)["dept"].unique())

# P10. One-hot encode dept
print(pd.get_dummies(emp["dept"], prefix="dept", dtype=int).head(3))

# P11. Find customers present in A but not in B (anti-join) + both (intersection)
A = pd.DataFrame({"cid": [1, 2, 3, 4]}); B = pd.DataFrame({"cid": [3, 4, 5]})
m = A.merge(B, on="cid", how="outer", indicator=True)
print(m.query("_merge == 'left_only'")["cid"].tolist(), m.query("_merge == 'both'")["cid"].tolist())

# P12. Replace outliers using IQR with the median
s = pd.Series([10, 12, 11, 13, 12, 300, 11])
q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
print(s.mask((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr), s.median()).tolist())

  dept name  salary
4  Fin    E     100
3   HR    D      65
0   IT    A      90
95
  name dept  salary
0    A   IT      90
1    B   IT      85
3    D   HR      65
4    E  Fin     100
  name  tenure band
0    A     4.0  2-5
1    B     4.8  2-5
2    C     2.5  2-5
3    D     5.1   5+
4    E     6.7   5+
5    F     1.9  0-2
6    G     1.0  0-2
7    H     3.3  2-5
dept
Fin    31.1
HR     29.8
IT     39.1
Name: salary, dtype: float64
  name     joined  cum
4    E 2017-05-05  100
3    D 2018-11-20  165
1    B 2019-03-15  250
0    A 2020-01-10  340
  dept name  salary  rk
4  Fin    E     100   1
5  Fin    F      95   2
3   HR    D      65   1
7   HR    H      62   2
2   HR    C      60   3
0   IT    A      90   1
1   IT    B      85   2
6   IT    G      70   3
joined
2017    1
2018    1
2019    1
2020    2
2021    1
2022    1
2023    1
Name: count, dtype: int64
<ArrowStringArray>
['IT', 'HR']
Length: 2, dtype: str
   dept_Fin  dept_HR  dept_IT
0         0        0        1
1         0        

## 25. Cheat Sheet 📌

| Task | Code |
|---|---|
| Create | `pd.DataFrame(dict)`, `pd.read_csv`, `pd.Series` |
| Inspect | `head tail info describe shape dtypes isna().sum() nunique value_counts` |
| Select | `df["c"]`, `df[["a","b"]]`, `df.loc[rows, cols]`, `df.iloc[i, j]`, `df[mask]`, `df.query("...")` |
| Modify | `df["new"]=...`, `assign`, `rename`, `drop`, `astype`, `replace`, `map` |
| Missing | `isna dropna fillna ffill bfill interpolate` |
| Duplicates | `duplicated drop_duplicates` |
| Strings / dates | `.str.*`, `.dt.*`, `to_datetime`, `resample` |
| Sort / rank | `sort_values sort_index nlargest rank` |
| Group | `groupby().agg / transform / filter / size / cumcount` |
| Reshape | `pivot pivot_table melt stack unstack crosstab get_dummies` |
| Combine | `merge join concat merge_asof` |
| Window | `shift diff pct_change rolling expanding ewm cumsum` |
| IO | `read_csv to_csv read_json read_excel read_parquet read_sql` |

### 🧭 Study Roadmap
1. Master **loc/iloc, boolean filters, groupby, merge** (≈ 80% of interview questions).
2. Practice **SQL ↔ Pandas translations** (`WHERE → filter`, `GROUP BY → groupby`, `JOIN → merge`, `OVER() → transform/rank/rolling`).
3. Understand **dtypes, missing values, copy semantics, performance**.
4. Next: **Matplotlib/Seaborn → scikit-learn**; for big data **Polars / Dask / PySpark**.

**Good luck with your interviews! 🚀**